In [8]:
# import os
# import zipfile

# # Use the path the search tool found
# zip_path = '/content/drive/MyDrive/Colab Notebooks/data.zip'
# extract_path = '/content/food_data'

# if not os.path.exists(extract_path):
#     with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#         zip_ref.extractall(extract_path)
#     print("✅ Unzip complete! Data is at /content/food_data")
# else:
#     print("Data already extracted.")

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import os
from torch.utils.data.sampler import SubsetRandomSampler
import matplotlib.pyplot as plt 

# --- 1. Setup Device (Ensure T4 GPU is on!) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 2. Advanced Transforms (The 60% Secret) ---
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 3. Final Dataset Class ---
class ColabFoodDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, is_test=False):
        self.df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        if not is_test:
            self.categories = sorted(self.df['label_name'].unique())
            self.label_map = {name: i for i, name in enumerate(self.categories)}

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        full_csv_path = str(self.df.loc[idx, 'image_path'])
        path_parts = full_csv_path.replace('\\', '/').split('/')

        # This matches your SUCCESS path perfectly
        actual_path = os.path.join(self.root_dir, path_parts[-2], path_parts[-1])

        image = Image.open(actual_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        if self.is_test:
            return image, self.df.loc[idx, 'original_index']

        label = self.label_map[self.df.loc[idx, 'label_name']]
        return image, torch.tensor(label)

# # --- 4. Initialize (Using the Verified Path) ---
# import os
# import shutil
# current_dir = os.getcwd()
# train_dir = os.path.join(current_dir, "Food10")
# csv_train = os.path.join(current_dir, 'train.csv.csv')

# train_ds = ColabFoodDataset(csv_train, train_dir, transform=train_transform)

# # Creating data indices for training and validation splits:
# dataset_size = len(train_ds)
# indices = list(range(dataset_size))
# train_num = int(0.7 * dataset_size)
# val_indices, train_indices = indices[train_num:], indices[:train_num]

# # Creating PT data samplers and loaders:
# train_sampler = SubsetRandomSampler(train_indices)
# valid_sampler = SubsetRandomSampler(val_indices)

# train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, sampler=train_sampler)
# validation_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, sampler=valid_sampler)

# # --- 6. Training Loop (xx Epochs) ---
# criterions = {"Cross Entropy": nn.CrossEntropyLoss()}
# lrs = {"0.00001":0.00001, "0.0001": 0.0001, "0.001": 0.001}


# criterion = nn.CrossEntropyLoss()

# i = 1
# for criterion_str, criterion in criterions.items():
#     for lr_str, lr in lrs.items():
        
#         # --- 5. Model: ResNet50 (Transfer Learning) ---
#         model = models.resnet50(weights='IMAGENET1K_V1')
#         model.fc = nn.Linear(model.fc.in_features, 10)
#         model = model.to(device)
#         optimizer = optim.Adam(model.parameters(), lr=lr)
#         train_loss = []
#         val_loss = []
#         for epoch in range(50):
#             # TRAIN STEP
#             model.train()
#             running_loss = 0.0
#             for images, labels in train_loader:
#                 images, labels = images.to(device), labels.to(device)
#                 optimizer.zero_grad()
#                 outputs = model(images)
#                 loss = criterion(outputs, labels)
#                 loss.backward()
#                 optimizer.step()
#                 running_loss += loss.item()
#             train_loss.append(running_loss / len(train_loader))

#             # VALIDATION STEP
#             model.eval()
#             running_loss = 0.0
#             with torch.no_grad():
#                 for images, labels in validation_loader:
#                     images, labels = images.to(device), labels.to(device)
#                     outputs = model(images)
#                     loss = criterion(outputs, labels)
#                     running_loss += loss.item()
#             val_loss.append(running_loss / len(validation_loader))

#         # PLOT
#         plt.figure(figsize=(8, 8)) 
#         plt.plot(range(50), train_loss, label='Training Loss') 
#         plt.plot(range(50), val_loss, label='Validation Loss') 
#         plt.legend(loc='lower right') 
#         plt.title(f'Training and Validation Loss for {lr_str} Learning Rate')
#         i += 1
#         print(f'Training and Validation Loss for {lr_str} Learning Rate')



# print("Starting High-Performance Training...")
# for epoch in range(50):
    
#     print(f"Epoch {epoch+1}/10 - Loss: {running_loss/len(train_loader):.4f}")


Using device: cpu


In [ ]:

# --- 4. Initialize (Using the Verified Path) ---
import os
import shutil
current_dir = os.getcwd()
train_dir = os.path.join(current_dir, "Food10")
csv_train = os.path.join(current_dir, 'train.csv.csv')

train_ds = ColabFoodDataset(csv_train, train_dir, transform=train_transform)

# Creating data indices for training and validation splits:
dataset_size = len(train_ds)
indices = list(range(dataset_size))
train_num = int(0.7 * dataset_size)
val_indices, train_indices = indices[train_num:], indices[:train_num]

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)

# --- 6. Training Loop (xx Epochs) ---
criterions = {"Cross Entropy": nn.CrossEntropyLoss()}
lrs = {"0.00001":0.00001, "0.0001": 0.0001, "0.001": 0.001}


criterion = nn.CrossEntropyLoss()
lr = 0.00001
model = models.resnet50(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
model.train()
for epoch in range(10):
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()



tensor([7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7])
tensor([7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7])
tensor([7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7])
tensor([7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7])
tensor([7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7])
tensor([7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7])
tensor([7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7])
tensor([7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7])
tensor([7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7,

KeyboardInterrupt: 

In [ ]:
# Kaggle
# 1. Define Test Transform (Clean, no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dir = os.path.join(current_dir, "FoodTest")
csv_test = os.path.join(current_dir, "test.csv")

# 2. Setup Test Dataset & Loader
test_ds = ColabFoodDataset(csv_test, test_dir, transform=test_transform, is_test=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

# 3. Predict
model.eval()
results = []
inv_label_map = {v: k for k, v in train_ds.label_map.items()}

print("Generating Kaggle predictions...")
with torch.no_grad():
    for images, indices in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        for idx, pred in zip(indices, predicted):
            results.append({
                "original_index": int(idx),
                "label_name": inv_label_map[pred.item()]
            })

# 4. Save and Download
submission_df = pd.DataFrame(results)
submission_df.sort_values("original_index", inplace=True) # Ensure correct order
submission_df.to_csv("high_perf_resnet_submission.csv", index=False)
print("✅ Saved! Look in the left sidebar 'Files' folder to download 'high_perf_resnet_submission.csv'.")

Generating Kaggle predictions...
tensor([[-0.3854, -0.4388,  0.2260, -0.3767,  0.0512, -0.1695, -0.3012, -0.3078,
         -0.4067,  0.2614],
        [-0.3939, -0.4648,  0.2348, -0.3977,  0.0464, -0.1745, -0.3079, -0.3253,
         -0.4189,  0.2671],
        [-0.3971, -0.4541,  0.2313, -0.3881,  0.0536, -0.1722, -0.3042, -0.3216,
         -0.4166,  0.2698],
        [-0.3990, -0.4464,  0.2304, -0.3857,  0.0608, -0.1837, -0.3019, -0.3185,
         -0.4082,  0.2649],
        [-0.3963, -0.4462,  0.2301, -0.3920,  0.0516, -0.1824, -0.3055, -0.3330,
         -0.4164,  0.2680],
        [-0.4067, -0.4699,  0.2411, -0.4035,  0.0589, -0.1730, -0.3043, -0.3339,
         -0.4299,  0.2760],
        [-0.4112, -0.4543,  0.2472, -0.4087,  0.0524, -0.1746, -0.3038, -0.3356,
         -0.4284,  0.2773],
        [-0.3810, -0.4412,  0.2169, -0.3780,  0.0528, -0.1689, -0.2992, -0.3103,
         -0.4044,  0.2636],
        [-0.3946, -0.4536,  0.2346, -0.4039,  0.0553, -0.1705, -0.3044, -0.3284,
         -0.41

KeyboardInterrupt: 